# SLR Word Diagnostics — Full 502 + Custom Subset

Standalone diagnostics for **ArSL word recognition** (does not modify `SLR_Diagnostics.ipynb` or training notebooks).

| Pipeline | Notebook | Key artifacts |
|----------|----------|---------------|
| Full 502 | `ArSL_Word_Training_v2.ipynb` | `arsl_word_sequences_v2_full.npz`, `arsl_v2_best.h5` |
| Custom ~45 | `ArSL_Word_Training_CustomWords.ipynb` | `arsl_custom_subset.npz`, `arsl_custom_best.h5` |

### What this checks

| Question | Cell |
|----------|------|
| Do NPZ / model / scaler files exist? | 2 |
| Is extraction healthy (shape, blanks, class counts)? | 4 |
| Are custom words covered in the cache? | 5 |
| Pose vs hand feature quality? | 6 |
| Full vs partial vs custom progress? | 7 |
| Model Top-1 / Top-5 / F1 on held-out test? | 9–10 |
| Worst classes & confusion pairs? | 11 |
| Best confidence threshold for deployment? | 12 |
| Kaggle upload & GPU readiness? | 13 |

**While extraction runs:** partial NPZ is audited automatically.


In [ ]:
# Cell 1 — Imports
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
print(f'TensorFlow {tf.__version__} | NumPy {np.__version__}')


In [ ]:
# Cell 2 — Paths & artifact inventory
PROJECT_ROOT = Path(r'M:/Term 10/Grad/SLR Main')
WORDS_DIR    = PROJECT_ROOT / 'Words' / 'ArSL Word (Arabic)'

ARTIFACTS = {
    'full_npz'      : WORDS_DIR / 'arsl_word_sequences_v2_full.npz',
    'partial_npz'   : WORDS_DIR / 'arsl_word_sequences_v2_partial.npz',
    'legacy_npz'    : WORDS_DIR / 'arsl_word_sequences_v2.npz',
    'custom_npz'    : WORDS_DIR / 'arsl_custom_subset.npz',
    'custom_csv'    : WORDS_DIR / 'KARSL-502_BasicWords.csv',
    'labels'        : WORDS_DIR / 'KARSL-502_Labels.txt',
    'full_model'    : WORDS_DIR / 'arsl_v2_best.h5',
    'full_scaler'   : WORDS_DIR / 'arsl_v2_scaler.npz',
    'full_classes'  : WORDS_DIR / 'arsl_v2_classes.csv',
    'custom_model'  : WORDS_DIR / 'arsl_custom_best.h5',
    'custom_scaler' : WORDS_DIR / 'arsl_custom_scaler.npz',
    'custom_classes': WORDS_DIR / 'arsl_custom_classes.csv',
}

EXPECTED_SEQ_LEN  = 48
EXPECTED_FEATURES = 258
EXPECTED_CLASSES  = 502
POSE_FEATURES     = 33 * 4
HAND_FEATURES     = 21 * 3
TEST_SIZE         = 0.4
RANDOM_STATE      = 42

def file_row(name, path):
    if not path.exists():
        return {'artifact': name, 'path': path.name, 'status': 'MISSING', 'size_mb': None}
    return {'artifact': name, 'path': path.name, 'status': 'OK',
            'size_mb': round(path.stat().st_size / 1e6, 2)}

inv = pd.DataFrame([file_row(k, v) for k, v in ARTIFACTS.items()])
print('=' * 62)
print('ARTIFACT INVENTORY')
print('=' * 62)
print(inv.to_string(index=False))
print(f'\nWords dir: {WORDS_DIR}')
if not any(ARTIFACTS[k].exists() for k in ('full_npz', 'partial_npz', 'legacy_npz')):
    print('\n⚠️  No NPZ yet — run ArSL_Word_Training_v2.ipynb Cell 6.')


In [ ]:
# Cell 3 — KArSL-502 labels
id_to_english, id_to_arabic = {}, {}
LABELS_FILE = ARTIFACTS['labels']

if LABELS_FILE.exists():
    with open(LABELS_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('SignID'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    sid = int(parts[0])
                    ar, en = parts[1].strip(), parts[2].strip()
                    cid = sid + 1
                    id_to_english[cid] = en if en and en not in ('?', '??', '') else str(cid)
                    id_to_arabic[cid]  = ar if ar and ar not in ('?', '??', '') else en
                except Exception:
                    pass
    print(f'Labels loaded: {len(id_to_english)} entries')
else:
    print('Labels file missing — using numeric IDs')

def word_name(cid):
    return id_to_english.get(int(cid), str(cid))


In [ ]:
# Cell 4 — NPZ health audit

def blank_frame_ratio(seq):
    return float(np.mean(np.all(seq == 0, axis=1)))

def audit_npz(path, label):
    if not path.exists():
        return None
    with np.load(str(path), mmap_mode='r') as d:
        if 'X' not in d.files or 'y' not in d.files:
            return {'label': label, 'error': 'missing X or y'}
        X, y = d['X'], d['y']
        n, seq, feat = X.shape
        classes, counts = np.unique(y, return_counts=True)
        idx = np.linspace(0, n - 1, min(n, 500), dtype=int)
        blank = np.mean([blank_frame_ratio(X[i]) for i in idx])
        return {
            'label': label, 'path': path.name, 'samples': int(n),
            'seq_len': int(seq), 'features': int(feat), 'classes': int(len(classes)),
            'min_per_class': int(counts.min()), 'max_per_class': int(counts.max()),
            'mean_per_class': round(float(counts.mean()), 1),
            'blank_frame_ratio': round(blank, 4),
            'nan_X': bool(np.isnan(X[: min(n, 1000)]).any()),
            'size_mb': round(path.stat().st_size / 1e6, 1),
            'seq_ok': seq == EXPECTED_SEQ_LEN, 'feat_ok': feat == EXPECTED_FEATURES,
            'class_ids': classes, 'counts': counts,
        }

NPZ_SOURCES = [
    ('FULL', ARTIFACTS['full_npz']),
    ('PARTIAL', ARTIFACTS['partial_npz']),
    ('LEGACY', ARTIFACTS['legacy_npz']),
    ('CUSTOM', ARTIFACTS['custom_npz']),
]

audits, rows = {}, []
for tag, p in NPZ_SOURCES:
    a = audit_npz(p, tag)
    if a is None:
        continue
    if 'error' in a:
        print(f'[{tag}] {a["error"]}')
        continue
    audits[tag] = a
    rows.append({k: v for k, v in a.items() if k not in ('class_ids', 'counts')})

if rows:
    print('=' * 62)
    print('NPZ HEALTH SUMMARY')
    print('=' * 62)
    print(pd.DataFrame(rows).to_string(index=False))
    for tag, a in audits.items():
        flags = []
        if not a['seq_ok']:
            flags.append(f'seq={a["seq_len"]} (want {EXPECTED_SEQ_LEN})')
        if not a['feat_ok']:
            flags.append(f'feat={a["features"]} (want {EXPECTED_FEATURES})')
        if a['classes'] < EXPECTED_CLASSES and tag in ('FULL', 'PARTIAL'):
            flags.append(f'{a["classes"]}/{EXPECTED_CLASSES} classes ({100*a["classes"]/EXPECTED_CLASSES:.0f}%)')
        if a['blank_frame_ratio'] > 0.3:
            flags.append(f'high blanks ({a["blank_frame_ratio"]:.1%})')
        if a['nan_X']:
            flags.append('NaN in X')
        sym = '✅' if not flags else '⚠️ '
        print(f'{sym} [{tag}]' + (' — ' + '; '.join(flags) if flags else ' healthy'))
else:
    print('No NPZ files found.')

PRIMARY_TAG = PRIMARY_NPZ = None
for tag in ('FULL', 'PARTIAL', 'LEGACY', 'CUSTOM'):
    if tag in audits:
        PRIMARY_TAG = tag
        PRIMARY_NPZ = dict(NPZ_SOURCES)[tag]
        break
if PRIMARY_NPZ:
    print(f'\nPrimary cache: {PRIMARY_TAG} → {PRIMARY_NPZ.name}')


In [ ]:
# Cell 5 — Custom word list coverage
print('=' * 62)
print('CUSTOM WORDS COVERAGE')
print('=' * 62)

CUSTOM_CSV = ARTIFACTS['custom_csv']
if not CUSTOM_CSV.exists():
    print(f'Missing {CUSTOM_CSV.name}')
else:
    cw = pd.read_csv(CUSTOM_CSV)
    target_ids = sorted(cw['class_id'].astype(int).unique())
    print(f'Words in CSV: {len(target_ids)}')
    ref = audits.get('FULL') or audits.get('PARTIAL') or audits.get('LEGACY')
    if ref is None:
        print('No reference NPZ — wait for extraction.')
    else:
        ref_tag = 'FULL' if 'FULL' in audits else ('PARTIAL' if 'PARTIAL' in audits else 'LEGACY')
        available = set(int(x) for x in ref['class_ids'])
        missing = [i for i in target_ids if i not in available]
        print(f'Reference: {ref_tag} ({ref["samples"]:,} samples, {ref["classes"]} classes)')
        print(f'  Covered: {len(target_ids)-len(missing)}/{len(target_ids)}')
        if missing:
            print(f'  Missing IDs: {missing}')
        preview = cw.drop_duplicates('class_id').copy()
        cnt_map = dict(zip(ref['class_ids'].astype(int), ref['counts']))
        preview['samples'] = preview['class_id'].map(lambda c: cnt_map.get(int(c), 0))
        cols = [c for c in ('class_id', 'english', 'category', 'samples') if c in preview.columns]
        print(preview[cols].head(25).to_string(index=False))


In [ ]:
# Cell 6 — Feature-stream quality + class distribution
def feature_stream_stats(X, sample_n=800):
    n = min(len(X), sample_n)
    sub = X[:n]
    pose = sub[:, :, :POSE_FEATURES]
    lh = sub[:, :, POSE_FEATURES:POSE_FEATURES + HAND_FEATURES]
    rh = sub[:, :, POSE_FEATURES + HAND_FEATURES:]
    def stream(name, arr):
        mag = np.linalg.norm(arr, axis=-1)
        return {'stream': name, 'mean_abs': round(float(np.mean(np.abs(arr))), 5),
                'std': round(float(np.std(arr)), 5),
                'zero_frame_rate': round(float(np.mean(mag < 1e-6)), 4)}
    return pd.DataFrame([stream('pose', pose), stream('left_hand', lh), stream('right_hand', rh)])

PATH_MAP = {'FULL': 'full_npz', 'PARTIAL': 'partial_npz', 'CUSTOM': 'custom_npz'}
for tag, key in PATH_MAP.items():
    if tag not in audits:
        continue
    with np.load(str(ARTIFACTS[key]), mmap_mode='r') as d:
        n = min(len(d['X']), 800)
        Xs = np.array(d['X'][:n])
    print(f'\n--- [{tag}] feature streams (n={n}) ---')
    print(feature_stream_stats(Xs).to_string(index=False))

if audits and PRIMARY_TAG:
    a = audits[PRIMARY_TAG]
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].hist(a['counts'], bins=30, color='steelblue', edgecolor='black', alpha=0.85)
    axes[0].set_title(f'[{PRIMARY_TAG}] Samples per class')
    axes[0].axvline(a['mean_per_class'], color='red', ls='--', label=f"mean={a['mean_per_class']}")
    axes[0].legend()
    names = [word_name(c) for c in a['class_ids']]
    order = np.argsort(a['counts'])[:25]
    axes[1].barh([names[i] for i in order], a['counts'][order], color='coral')
    axes[1].set_title(f'[{PRIMARY_TAG}] 25 lowest-count classes')
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 7 — Pipeline comparison
def pipeline_row(name, prefix):
    npz_key = f'{prefix}_npz' if prefix != 'partial' else 'partial_npz'
    aud_tag = prefix.upper() if prefix != 'partial' else 'PARTIAL'
    aud = audits.get(aud_tag)
    return {
        'pipeline': name,
        'npz': '✓' if ARTIFACTS[npz_key].exists() else '—',
        'samples': aud['samples'] if aud else '—',
        'classes': aud['classes'] if aud else '—',
        'model': '✓' if prefix != 'partial' and ARTIFACTS.get(f'{prefix}_model', Path()).exists() else '—',
        'scaler': '✓' if prefix != 'partial' and ARTIFACTS.get(f'{prefix}_scaler', Path()).exists() else '—',
    }

print('=' * 62)
print('PIPELINE COMPARISON')
print('=' * 62)
print(pd.DataFrame([
    pipeline_row('Full 502 (v2)', 'full'),
    pipeline_row('Partial (in progress)', 'partial'),
    pipeline_row('Custom subset', 'custom'),
]).to_string(index=False))

if 'PARTIAL' in audits and 'FULL' not in audits:
    p = audits['PARTIAL']
    print(f'\nExtraction progress: ~{100*p["classes"]/EXPECTED_CLASSES:.0f}% ({p["classes"]}/502 classes)')


In [ ]:
# Cell 8 — Evaluation helpers

def apply_scaler(X, scaler_path):
    if not scaler_path.exists():
        return X, False
    s = np.load(str(scaler_path))
    sh = X.shape
    flat = X.reshape(-1, sh[-1])
    flat = (flat - s['mean']) / s['scale']
    return flat.reshape(sh).astype(np.float32), True

def eval_word_model(profile):
    if profile == 'full':
        npz_p = ARTIFACTS['full_npz'] if ARTIFACTS['full_npz'].exists() else ARTIFACTS['partial_npz']
    else:
        npz_p = ARTIFACTS['custom_npz']
    model_p = ARTIFACTS[f'{profile}_model']
    scaler_p = ARTIFACTS[f'{profile}_scaler']
    classes_p = ARTIFACTS[f'{profile}_classes']

    result = {'profile': profile.upper(), 'status': 'skip'}
    if not npz_p.exists():
        result['reason'] = f'NPZ missing: {npz_p.name}'
        return result
    if not model_p.exists():
        result['reason'] = f'Model missing: {model_p.name}'
        return result

    d = np.load(str(npz_p))
    X, y_raw = d['X'], d['y']
    X, scaled = apply_scaler(X, scaler_p)
    le = LabelEncoder()
    y_enc = le.fit_transform(y_raw)
    if classes_p.exists():
        cdf = pd.read_csv(classes_p)
        name_map = dict(zip(cdf['karsl_class_id'].astype(int), cdf['english']))
        labels = [name_map.get(int(c), word_name(c)) for c in le.classes_]
    else:
        labels = [word_name(c) for c in le.classes_]
    try:
        _, X_te, _, y_te = train_test_split(
            X, y_enc, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_enc)
    except ValueError:
        _, X_te, _, y_te = train_test_split(
            X, y_enc, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    model = tf.keras.models.load_model(str(model_p))
    proba = model.predict(X_te, verbose=0)
    y_pred = np.argmax(proba, axis=1)
    top1 = accuracy_score(y_te, y_pred)
    top5 = float(np.mean([y_te[i] in np.argsort(proba[i])[-5:] for i in range(len(y_te))]))
    macro_f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
    result.update({'status': 'ok', 'npz': npz_p.name, 'model': model_p.name, 'scaled': scaled,
                   'test_n': len(y_te), 'classes': len(le.classes_), 'top1': top1, 'top5': top5,
                   'macro_f1': macro_f1, 'y_te': y_te, 'y_pred': y_pred, 'proba': proba, 'labels': labels})
    return result

print('Eval helpers ready.')


In [ ]:
# Cell 9 — FULL 502-word model
eval_full = eval_word_model('full')
print('=' * 62)
print('FULL MODEL (502-word pipeline)')
print('=' * 62)
if eval_full['status'] != 'ok':
    print(f"Skip: {eval_full.get('reason')}")
else:
    print(f"  NPZ     : {eval_full['npz']}")
    print(f"  Scaler  : {'yes' if eval_full['scaled'] else 'NO — metrics unreliable'}")
    print(f"  Top-1   : {eval_full['top1']*100:.2f}%")
    print(f"  Top-5   : {eval_full['top5']*100:.2f}%")
    print(f"  Macro F1: {eval_full['macro_f1']:.4f}")


In [ ]:
# Cell 10 — CUSTOM subset model
eval_custom = eval_word_model('custom')
print('=' * 62)
print('CUSTOM MODEL (subset pipeline)')
print('=' * 62)
if eval_custom['status'] != 'ok':
    print(f"Skip: {eval_custom.get('reason')}")
else:
    print(f"  Top-1   : {eval_custom['top1']*100:.2f}%")
    print(f"  Top-5   : {eval_custom['top5']*100:.2f}%")
    print(f"  Macro F1: {eval_custom['macro_f1']:.4f}")
    print('\n' + classification_report(
        eval_custom['y_te'], eval_custom['y_pred'],
        target_names=eval_custom['labels'], zero_division=0))


In [ ]:
# Cell 11 — Confusions & weak classes

def plot_confusions(ev, title, max_labels=40):
    if ev.get('status') != 'ok':
        print(f'Skip {title}: {ev.get("reason", "n/a")}')
        return
    y_te, y_pred, labels = ev['y_te'], ev['y_pred'], ev['labels']
    n = len(labels)
    cm = confusion_matrix(y_te, y_pred)
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    order = np.argsort(recall)
    show = min(n, max_labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, max(5, show * 0.22)))
    w = order[:show]
    axes[0].barh([labels[i] for i in w], recall[w], color='indianred')
    axes[0].set_xlim(0, 1.05)
    axes[0].set_title(f'{title} — worst {show} by recall')
    pairs = [(labels[i], labels[j], int(cm[i,j])) for i in range(n) for j in range(n) if i != j and cm[i,j]]
    pairs.sort(key=lambda x: -x[2])
    if pairs:
        top = pairs[:15]
        axes[1].barh([f'{a}→{b}' for a,b,_ in top], [c for *_,c in top], color='steelblue')
        axes[1].set_title(f'{title} — top confusions')
    plt.tight_layout()
    plt.show()

plot_confusions(eval_full, 'FULL 502')
plot_confusions(eval_custom, 'CUSTOM', max_labels=45)


In [ ]:
# Cell 12 — Confidence threshold sweep
def threshold_sweep(ev, name):
    if ev.get('status') != 'ok':
        print(f'[{name}] skip')
        return
    conf = np.max(ev['proba'], axis=1)
    ok = ev['y_pred'] == ev['y_te']
    rows = []
    for t in np.arange(0.3, 0.96, 0.05):
        m = conf >= t
        rows.append({'thr': round(t,2), 'coverage': round(m.mean(),3),
                     'accuracy': round(ok[m].mean(),3) if m.any() else 0})
    df = pd.DataFrame(rows)
    print(f'\n[{name}]')
    print(df.to_string(index=False))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['thr'], df['accuracy'], 'o-', label='Accuracy')
    ax.plot(df['thr'], df['coverage'], 's--', label='Coverage')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{name} threshold tradeoff')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print('THRESHOLD SWEEP')
threshold_sweep(eval_full, 'FULL')
threshold_sweep(eval_custom, 'CUSTOM')


In [ ]:
# Cell 13 — Readiness checklist
checks = []
def chk(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})

ref = audits.get('FULL') or audits.get('PARTIAL')
chk('NPZ cache', ref is not None, f"{ref['samples'] if ref else 0} samples")
chk('Shape 48×258', ref and ref.get('seq_ok') and ref.get('feat_ok'), '')
chk('502 classes (full)', audits.get('FULL') and audits['FULL']['classes'] >= 502,
    f"{audits['FULL']['classes'] if 'FULL' in audits else 'n/a'} classes")
chk('Full model', ARTIFACTS['full_model'].exists(), ARTIFACTS['full_model'].name)
chk('Full scaler', ARTIFACTS['full_scaler'].exists(), 'inference')
chk('Custom NPZ', ARTIFACTS['custom_npz'].exists(), 'CustomWords Cell 5')
chk('Custom model', ARTIFACTS['custom_model'].exists(), 'app demo')
gpus = tf.config.list_physical_devices('GPU')
chk('GPU', len(gpus) > 0, gpus[0].name if gpus else 'CPU only')

print('=' * 62)
print('READINESS')
print('=' * 62)
print(pd.DataFrame(checks).to_string(index=False))
if ref:
    print(f"\nEst. RAM for full X: ~{ref['samples']*EXPECTED_SEQ_LEN*EXPECTED_FEATURES*4/1e6:.0f} MB")


### Interpretation guide

| Symptom | Cause | Action |
|---------|-------|--------|
| PARTIAL < 502 classes | Extraction running | Wait for v2 Cell 6 |
| High blank_frame_ratio | Bad/occluded videos | Filter in extraction |
| Full Top-1 low, custom high | 502-class is harder | Use custom model in app |
| Model skip | Not trained yet | Run training cells |
| No scaler | Preprocess not run | Re-run preprocess cell |
| Low coverage @ 0.75 thr | Uncertain predictions | Lower threshold |
